# Maze Crawler — Imitation Learning training (Phase B)

**Before running:**
1. Upload the `bunterr/` folder (~50 MB of replay JSONs) as a Kaggle Dataset, and attach it to this notebook. The notebook auto-detects it at `/kaggle/input/<your-dataset>/bunterr/` or `/kaggle/input/<your-dataset>/`.
2. **Settings → Accelerator → GPU T4 x1 (or P100)**.
3. Run all cells. Training takes ~3-5 minutes. Output is `/kaggle/working/weights.pt`.

This notebook bundles the encoder inline so no separate upload is needed.


In [1]:
# === Setup ===
import os, sys, glob, json, time, math
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)
if DEVICE == 'cpu':
    print('!!! GPU NOT ENABLED — flip Accelerator to GPU and re-run !!!')


torch 2.10.0+cu128 | CUDA available: True
Using device: cuda


In [2]:
# === Encoder (bundled inline) ===
"""
Maze Crawler IL — observation encoder + action vocabulary.

For each (replay step, decision-making unit), produce:
  - grid:    float32 [C=18, H=21, W=21] centred on the unit
  - scalars: float32 [8]  per-unit features
  - mask:    bool   [A]   which actions are valid for this unit type

Encoder uses the FOGGED per-player observation, the same view the bot sees at
inference time (no information leak from globals).
"""
import json
import os
import numpy as np

GRID_HW = 21
HALF = GRID_HW // 2
NUM_CHANNELS = 18
NUM_SCALARS = 8
WIDTH = 20  # crawl width

# --- Action vocabulary (canonical, fixed order) -----------------------------
ACTIONS = (
    "IDLE",
    "NORTH", "SOUTH", "EAST", "WEST",
    "JUMP_NORTH", "JUMP_SOUTH", "JUMP_EAST", "JUMP_WEST",
    "BUILD_SCOUT_NORTH",  "BUILD_SCOUT_SOUTH",  "BUILD_SCOUT_EAST",  "BUILD_SCOUT_WEST",
    "BUILD_WORKER_NORTH", "BUILD_WORKER_SOUTH", "BUILD_WORKER_EAST", "BUILD_WORKER_WEST",
    "BUILD_MINER_NORTH",  "BUILD_MINER_SOUTH",  "BUILD_MINER_EAST",  "BUILD_MINER_WEST",
    "BUILD_NORTH", "BUILD_SOUTH", "BUILD_EAST", "BUILD_WEST",
    "REMOVE_NORTH", "REMOVE_SOUTH", "REMOVE_EAST", "REMOVE_WEST",
    "TRANSFORM",
    "TRANSFER_NORTH", "TRANSFER_SOUTH", "TRANSFER_EAST", "TRANSFER_WEST",
)
NUM_ACTIONS = len(ACTIONS)              # 34
ACT_TO_IDX = {a: i for i, a in enumerate(ACTIONS)}

FACTORY, SCOUT, WORKER, MINER = 0, 1, 2, 3

# What each unit type is allowed to do
_MOVES = ("IDLE", "NORTH", "SOUTH", "EAST", "WEST")
_TRANSFERS = ("TRANSFER_NORTH", "TRANSFER_SOUTH", "TRANSFER_EAST", "TRANSFER_WEST")
_VALID = {
    FACTORY: ("IDLE",
              "NORTH", "SOUTH", "EAST", "WEST",
              "JUMP_NORTH", "JUMP_SOUTH", "JUMP_EAST", "JUMP_WEST",
              "BUILD_SCOUT_NORTH",  "BUILD_SCOUT_SOUTH",  "BUILD_SCOUT_EAST",  "BUILD_SCOUT_WEST",
              "BUILD_WORKER_NORTH", "BUILD_WORKER_SOUTH", "BUILD_WORKER_EAST", "BUILD_WORKER_WEST",
              "BUILD_MINER_NORTH",  "BUILD_MINER_SOUTH",  "BUILD_MINER_EAST",  "BUILD_MINER_WEST",
              *_TRANSFERS),
    SCOUT:   (*_MOVES, *_TRANSFERS),
    WORKER:  (*_MOVES,
              "BUILD_NORTH", "BUILD_SOUTH", "BUILD_EAST", "BUILD_WEST",
              "REMOVE_NORTH", "REMOVE_SOUTH", "REMOVE_EAST", "REMOVE_WEST",
              *_TRANSFERS),
    MINER:   (*_MOVES, "TRANSFORM", *_TRANSFERS),
}

def valid_mask(unit_type: int) -> np.ndarray:
    m = np.zeros(NUM_ACTIONS, dtype=bool)
    for a in _VALID.get(unit_type, ("IDLE",)):
        m[ACT_TO_IDX[a]] = True
    return m

def canonical_action(a: str) -> str:
    """`BUILD_SCOUT` etc default to north per the engine."""
    if a in ("BUILD_SCOUT", "BUILD_WORKER", "BUILD_MINER"):
        return a + "_NORTH"
    return a

# --- Encoder ---------------------------------------------------------------
def _max_energy(rtype: int) -> float:
    return (1.0, 100.0, 300.0, 500.0)[rtype] if rtype < 4 else 1.0

def _pk(k):
    a, b = k.split(",")
    return int(a), int(b)

def encode(obs, unit_uid, width=WIDTH):
    """Encode `obs` from the perspective of `unit_uid`. Returns (grid, scalars, mask)."""
    unit = obs["robots"][unit_uid]
    utype, ucol, urow, uenergy, uowner = unit[0], unit[1], unit[2], unit[3], unit[4]
    move_cd  = unit[5] if len(unit) > 5 else 0
    jump_cd  = unit[6] if len(unit) > 6 else 0
    build_cd = unit[7] if len(unit) > 7 else 0

    south = obs.get("southBound", 0)
    north = obs.get("northBound", south + 19)
    walls = obs.get("walls") or []
    step  = obs.get("step", 0)

    grid = np.zeros((NUM_CHANNELS, GRID_HW, GRID_HW), dtype=np.float32)

    # Walls + discovered mask
    for gr in range(GRID_HW):
        for gc in range(GRID_HW):
            ac = ucol + (gc - HALF)
            ar = urow + (gr - HALF)
            if not (0 <= ac < width and south <= ar <= north):
                continue
            idx = (ar - south) * width + ac
            if 0 <= idx < len(walls):
                w = walls[idx]
                if w != -1:
                    grid[4, gr, gc] = 1.0
                    if w & 1: grid[0, gr, gc] = 1.0
                    if w & 2: grid[1, gr, gc] = 1.0
                    if w & 4: grid[2, gr, gc] = 1.0
                    if w & 8: grid[3, gr, gc] = 1.0

    # Robots (fogged: own always visible + enemies in vision)
    me_owner = uowner
    for ruid, r in obs.get("robots", {}).items():
        rt, rc, rr_, re_, ro = r[0], r[1], r[2], r[3], r[4]
        gc = rc - ucol + HALF
        gr = rr_ - urow + HALF
        if not (0 <= gc < GRID_HW and 0 <= gr < GRID_HW):
            continue
        if ro == me_owner:
            if   rt == FACTORY: grid[5, gr, gc] = 1.0
            elif rt == SCOUT:   grid[6, gr, gc] = 1.0
            elif rt == WORKER:  grid[7, gr, gc] = 1.0
            elif rt == MINER:   grid[8, gr, gc] = 1.0
        else:
            grid[9, gr, gc] = 1.0
        if rt == FACTORY:
            grid[14, gr, gc] = min(1.0, re_ / 1000.0)  # normalize factory by start
        else:
            mx = _max_energy(rt) or 1.0
            grid[14, gr, gc] = max(0.0, min(1.0, re_ / mx))
        if ruid == unit_uid:
            grid[15, gr, gc] = 1.0

    # Friendly / enemy mines
    for k, m in (obs.get("mines") or {}).items():
        c, r = _pk(k)
        gc = c - ucol + HALF; gr = r - urow + HALF
        if 0 <= gc < GRID_HW and 0 <= gr < GRID_HW:
            owner = m[2] if len(m) > 2 else -1
            if owner == me_owner: grid[10, gr, gc] = 1.0
            else:                 grid[11, gr, gc] = 1.0
    # Mining nodes (visible)
    for k in (obs.get("miningNodes") or {}):
        c, r = _pk(k)
        gc = c - ucol + HALF; gr = r - urow + HALF
        if 0 <= gc < GRID_HW and 0 <= gr < GRID_HW:
            grid[12, gr, gc] = 1.0
    # Crystals (visible)
    for k in (obs.get("crystals") or {}):
        c, r = _pk(k)
        gc = c - ucol + HALF; gr = r - urow + HALF
        if 0 <= gc < GRID_HW and 0 <= gr < GRID_HW:
            grid[13, gr, gc] = 1.0

    # Broadcast scalars
    grid[16, :, :] = min(1.0, step / 500.0)
    # scroll interval interpolates 10 -> 2 by step 450
    si = 10.0 - 8.0 * min(1.0, step / 450.0)
    grid[17, :, :] = 1.0 / max(si, 1.0)

    scalars = np.zeros(NUM_SCALARS, dtype=np.float32)
    scalars[0] = utype / 3.0
    scalars[1] = min(1.0, uenergy / 1000.0) if utype == FACTORY else min(1.0, uenergy / (_max_energy(utype) or 1.0))
    scalars[2] = min(1.0, move_cd / 2.0)
    scalars[3] = min(1.0, jump_cd / 20.0)  if utype == FACTORY else 0.0
    scalars[4] = min(1.0, build_cd / 10.0) if utype == FACTORY else 0.0
    scalars[5] = max(0.0, min(1.0, (urow - south) / 20.0))
    scalars[6] = min(1.0, step / 500.0)
    scalars[7] = float(uowner)  # 0 or 1, lets the model condition on side

    mask = valid_mask(utype)
    return grid, scalars, mask

# --- Replay parser ---------------------------------------------------------
def find_bunterr_idx(replay_dict):
    for i, ag in enumerate(replay_dict.get("info", {}).get("Agents", [])):
        if str(ag.get("Name", "")).lower().startswith("bunterr"):
            return i
    return None

def iter_examples(replay_path, want_player_name_prefix="bunterr"):
    """Yields (game_id, t, uid, obs, action_idx, unit_type) per friendly unit action."""
    d = json.load(open(replay_path))
    if want_player_name_prefix:
        bi = None
        for i, ag in enumerate(d.get("info", {}).get("Agents", [])):
            if str(ag.get("Name", "")).lower().startswith(want_player_name_prefix):
                bi = i; break
        if bi is None:
            return
    else:
        bi = 0
    game_id = os.path.basename(replay_path).split(".")[0]
    steps = d["steps"]
    for t in range(len(steps)):
        obs = steps[t][bi]["observation"]
        action = steps[t][bi].get("action")
        if not isinstance(action, dict):
            continue
        robots = obs.get("robots") or {}
        for uid, a_str in action.items():
            if uid not in robots:
                continue
            a_str = canonical_action(a_str)
            if a_str not in ACT_TO_IDX:
                continue                                # unknown action -> skip
            unit_type = robots[uid][0]
            yield (game_id, t, uid, obs, ACT_TO_IDX[a_str], unit_type)


In [3]:
# === Find the bunterr replay folder in the attached dataset ===
import glob
candidates = (
    glob.glob('/kaggle/input/datasets/alihamzapeenek/bunterr/*.json')
    + glob.glob('/kaggle/input/*/*.json')
    + glob.glob('/kaggle/input/*/bunterr*.json')
)

# group by directory and pick the dir with the most json files
by_dir = defaultdict(list)
for p in candidates: by_dir[os.path.dirname(p)].append(p)
if not by_dir:
    raise RuntimeError('No replay JSONs found under /kaggle/input/. Did you attach the dataset?')
REPLAY_DIR = max(by_dir, key=lambda k: len(by_dir[k]))
replays = sorted(by_dir[REPLAY_DIR])
print('Using replay dir:', REPLAY_DIR)
print('Number of replay files:', len(replays))


Using replay dir: /kaggle/input/datasets/alihamzapeenek/bunterr
Number of replay files: 106


In [4]:
# === Train/val split BY GAME so we don't leak within-game context ===
rng = np.random.default_rng(0)
perm = rng.permutation(len(replays))
n_val = max(1, len(replays) // 10)
val_files = [replays[i] for i in perm[:n_val]]
train_files = [replays[i] for i in perm[n_val:]]
print(f'train games: {len(train_files)}  val games: {len(val_files)}')


train games: 96  val games: 10


In [5]:
# === Encode all examples to numpy in memory ===
def encode_files(files):
    grids, scals, masks, labels, types = [], [], [], [], []
    for f in files:
        for gid, t, uid, obs, a_idx, utype in iter_examples(f):
            g, s, m = encode(obs, uid)
            grids.append(g.astype(np.float16))   # save memory
            scals.append(s.astype(np.float32))
            masks.append(m.astype(np.bool_))
            labels.append(a_idx)
            types.append(utype)
    return (np.stack(grids), np.stack(scals), np.stack(masks),
            np.array(labels, dtype=np.int64), np.array(types, dtype=np.int64))
t0 = time.time()
Xg_tr, Xs_tr, Mk_tr, y_tr, ty_tr = encode_files(train_files)
Xg_va, Xs_va, Mk_va, y_va, ty_va = encode_files(val_files)
print(f'encoded in {time.time()-t0:.1f}s')
print('train:', Xg_tr.shape, Xs_tr.shape, Mk_tr.shape, y_tr.shape, '  bytes:', Xg_tr.nbytes//(1024*1024), 'MB')
print('val:  ', Xg_va.shape)
# action distribution
cnt = Counter(y_tr.tolist())
print('top actions train:', [(ACTIONS[a], n) for a,n in cnt.most_common(8)])


encoded in 18.2s
train: (38623, 18, 21, 21) (38623, 8) (38623, 34) (38623,)   bytes: 584 MB
val:   (3149, 18, 21, 21)
top actions train: [('NORTH', 14009), ('IDLE', 14006), ('EAST', 2425), ('WEST', 2412), ('SOUTH', 1610), ('REMOVE_NORTH', 1485), ('JUMP_NORTH', 1247), ('TRANSFER_NORTH', 596)]


In [6]:
# === Class weights for weighted cross-entropy (inverse-sqrt frequency) ===
freq = np.zeros(NUM_ACTIONS, dtype=np.float64)
for a, n in cnt.items(): freq[a] = n
freq = np.where(freq > 0, freq, 1.0)
weights = 1.0 / np.sqrt(freq)
weights = weights / weights.mean()        # normalize so loss scale stays familiar
weights = np.clip(weights, 0.1, 20.0).astype(np.float32)
print('class weights min/median/max:', weights.min(), float(np.median(weights)), weights.max())


class weights min/median/max: 0.1 1.6124788522720337 1.6124789


In [7]:
# === Model: small CNN trunk + scalars MLP + masked logits ===
class ResBlock(nn.Module):
    def __init__(self, c): super().__init__(); self.c1=nn.Conv2d(c,c,3,padding=1); self.b1=nn.BatchNorm2d(c); self.c2=nn.Conv2d(c,c,3,padding=1); self.b2=nn.BatchNorm2d(c)
    def forward(self,x): return F.relu(x + self.b2(self.c2(F.relu(self.b1(self.c1(x))))))

class CrawlNet(nn.Module):
    def __init__(self, in_ch=NUM_CHANNELS, hidden=64, n_actions=NUM_ACTIONS, n_scalars=NUM_SCALARS):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(in_ch, hidden, 3, padding=1), nn.BatchNorm2d(hidden), nn.ReLU())
        self.blocks = nn.Sequential(*[ResBlock(hidden) for _ in range(3)])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Linear(hidden + n_scalars, 128), nn.ReLU(),
            nn.Linear(128, n_actions),
        )
    def forward(self, grid, scalars, mask):
        h = self.blocks(self.stem(grid))                       # [B, hidden, 21,21]
        h = self.pool(h).flatten(1)                            # [B, hidden]
        z = torch.cat([h, scalars], dim=1)                     # [B, hidden+nsc]
        logits = self.head(z)                                  # [B, n_actions]
        logits = logits.masked_fill(~mask, float('-inf'))      # zero out invalid actions
        return logits

model = CrawlNet().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'model params: {n_params:,}')


model params: 246,626


In [8]:
# === DataLoaders ===
def make_loader(Xg, Xs, Mk, y, batch_size=256, shuffle=True):
    ds = TensorDataset(torch.from_numpy(Xg), torch.from_numpy(Xs), torch.from_numpy(Mk), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader(Xg_tr, Xs_tr, Mk_tr, y_tr, 256, True)
val_loader   = make_loader(Xg_va, Xs_va, Mk_va, y_va, 512, False)
print('train batches:', len(train_loader), 'val batches:', len(val_loader))


train batches: 151 val batches: 7


In [9]:
# === Training loop ===
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
w_tensor = torch.from_numpy(weights).to(DEVICE)
crit = nn.CrossEntropyLoss(weight=w_tensor)

def evaluate(loader):
    model.eval()
    correct = total = 0
    per_action_correct = np.zeros(NUM_ACTIONS, dtype=np.int64)
    per_action_total   = np.zeros(NUM_ACTIONS, dtype=np.int64)
    with torch.no_grad():
        for g, s, m, y in loader:
            g = g.to(DEVICE, dtype=torch.float32, non_blocking=True)
            s = s.to(DEVICE, non_blocking=True)
            m = m.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            logits = model(g, s, m)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item(); total += y.numel()
            for a in range(NUM_ACTIONS):
                mask_a = (y == a)
                per_action_total[a]   += mask_a.sum().item()
                per_action_correct[a] += ((pred == y) & mask_a).sum().item()
    return correct/total, per_action_correct, per_action_total

BASELINE = (y_va == ACT_TO_IDX['IDLE']).mean()
print(f'val baseline (always-IDLE accuracy): {BASELINE:.3f}')

best_val = 0.0
for epoch in range(20):
    model.train()
    t0 = time.time(); tot_loss = 0.0; tot_n = 0
    for g, s, m, y in train_loader:
        g = g.to(DEVICE, dtype=torch.float32, non_blocking=True)
        s = s.to(DEVICE, non_blocking=True)
        m = m.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(g, s, m)
        loss = crit(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        tot_loss += loss.item()*y.numel(); tot_n += y.numel()
    sched.step()
    val_acc, pa_c, pa_t = evaluate(val_loader)
    if val_acc > best_val:
        best_val = val_acc
        torch.save({'model_state': model.state_dict(),
                    'num_channels': NUM_CHANNELS, 'num_scalars': NUM_SCALARS,
                    'num_actions': NUM_ACTIONS, 'actions': list(ACTIONS)},
                   '/kaggle/working/weights.pt')
        tag = ' (saved)'
    else:
        tag = ''
    print(f'epoch {epoch:2d}  loss {tot_loss/tot_n:.4f}  val_acc {val_acc:.3f}  baseline {BASELINE:.3f}  time {time.time()-t0:.1f}s{tag}')


val baseline (always-IDLE accuracy): 0.412
epoch  0  loss 1.3626  val_acc 0.712  baseline 0.412  time 10.6s (saved)
epoch  1  loss 0.9053  val_acc 0.767  baseline 0.412  time 9.2s (saved)
epoch  2  loss 0.7668  val_acc 0.773  baseline 0.412  time 9.2s (saved)
epoch  3  loss 0.6783  val_acc 0.752  baseline 0.412  time 9.2s
epoch  4  loss 0.6297  val_acc 0.790  baseline 0.412  time 9.3s (saved)
epoch  5  loss 0.5933  val_acc 0.788  baseline 0.412  time 9.3s
epoch  6  loss 0.5625  val_acc 0.789  baseline 0.412  time 9.4s
epoch  7  loss 0.5374  val_acc 0.800  baseline 0.412  time 9.4s (saved)
epoch  8  loss 0.5129  val_acc 0.788  baseline 0.412  time 9.5s
epoch  9  loss 0.4953  val_acc 0.790  baseline 0.412  time 9.6s
epoch 10  loss 0.4783  val_acc 0.783  baseline 0.412  time 9.6s
epoch 11  loss 0.4616  val_acc 0.807  baseline 0.412  time 9.8s (saved)
epoch 12  loss 0.4503  val_acc 0.808  baseline 0.412  time 10.0s (saved)
epoch 13  loss 0.4393  val_acc 0.810  baseline 0.412  time 9.8s (sa

In [10]:
# === Final per-action accuracy on the val set (the metric that really matters) ===
val_acc, pa_c, pa_t = evaluate(val_loader)
print(f'overall val acc {val_acc:.3f}  baseline {BASELINE:.3f}\n')
print(f"{'action':22s} {'count':>6s} {'acc':>6s}")
for a in range(NUM_ACTIONS):
    if pa_t[a] == 0: continue
    print(f"{ACTIONS[a]:22s} {pa_t[a]:>6d} {pa_c[a]/pa_t[a]:>6.3f}")


overall val acc 0.814  baseline 0.412

action                  count    acc
IDLE                     1296  0.970
NORTH                    1157  0.866
SOUTH                      72  0.222
EAST                      150  0.320
WEST                      144  0.340
JUMP_NORTH                 79  0.329
BUILD_WORKER_NORTH         10  0.800
BUILD_MINER_NORTH          14  1.000
BUILD_MINER_SOUTH           1  0.000
BUILD_MINER_EAST            4  1.000
BUILD_MINER_WEST            7  1.000
REMOVE_NORTH               95  0.526
TRANSFORM                  69  1.000
TRANSFER_NORTH             51  0.235


In [11]:
# === Sanity-check: load saved weights, run one forward pass ===
ckpt = torch.load('/kaggle/working/weights.pt', map_location=DEVICE, weights_only=False)
m2 = CrawlNet().to(DEVICE); m2.load_state_dict(ckpt['model_state']); m2.eval()
with torch.no_grad():
    g = torch.from_numpy(Xg_va[:4]).to(DEVICE, dtype=torch.float32)
    s = torch.from_numpy(Xs_va[:4]).to(DEVICE)
    mk = torch.from_numpy(Mk_va[:4]).to(DEVICE)
    logits = m2(g, s, mk)
    pred = logits.argmax(1).cpu().numpy()
    print('first 4 val predictions:', [ACTIONS[a] for a in pred])
    print('first 4 val labels:     ', [ACTIONS[a] for a in y_va[:4]])
print('weights.pt size:', os.path.getsize('/kaggle/working/weights.pt')/1024, 'KB')
print('\nNEXT: download /kaggle/working/weights.pt and drop it in your Maze Crawler folder.')


first 4 val predictions: ['JUMP_NORTH', 'JUMP_NORTH', 'EAST', 'EAST']
first 4 val labels:      ['JUMP_NORTH', 'EAST', 'EAST', 'EAST']
weights.pt size: 984.37890625 KB

NEXT: download /kaggle/working/weights.pt and drop it in your Maze Crawler folder.
